# Irene neural avatar test (free Google Colab)

This notebook runs the official LivePortrait human model on a temporary Colab GPU. It uploads a source photo, animates it with a short driving clip, previews the result, and lets you download the MP4.

Colab sessions are temporary and free GPU access is not guaranteed. Do not upload anything you do not want processed by Google Colab.

In [1]:
import os
import shutil
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU was assigned. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')
print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))

GPU available: True
GPU: Tesla T4
Free disk (GB): 61.8


In [2]:
!git clone -q --depth 1 https://github.com/KlingTeam/LivePortrait.git /content/LivePortrait
%cd /content/LivePortrait
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[cli]"

fatal: destination path '/content/LivePortrait' already exists and is not an empty directory.
/content/LivePortrait


In [3]:
%cd /content/LivePortrait
!huggingface-cli download KlingTeam/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"
print('Pretrained weights are ready.')

/content/LivePortrait
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 20 files: 100% 20/20 [00:00<00:00, 934.79it/s]
/content/LivePortrait/pretrained_weights
Pretrained weights are ready.


In [4]:
from google.colab import files
import os

print('Choose Irene photo (PNG or JPG).')
uploaded = files.upload()
original_filename = next(iter(uploaded))

# Rename the uploaded file to a simpler name without spaces or special characters
new_filename = 'source.' + original_filename.split('.')[-1]
source_path = os.path.join('/content/LivePortrait/', new_filename) # Corrected path
os.rename(os.path.join('/content/LivePortrait/', original_filename), source_path)

print('Source photo:', source_path)

Choose Irene photo (PNG or JPG).


Saving Irene .PNG to Irene  (4).PNG
Source photo: /content/LivePortrait/source.PNG


In [13]:
%cd /content/LivePortrait
!python inference.py -s "{source_path}" -d assets/examples/driving/d9.mp4 -o animations --flag_crop_driving_video --driving_option expression-friendly

/content/LivePortrait
[20:48:17] Load appearance_feature_extractor from    ]8;id=410398;file:///content/LivePortrait/src/live_portrait_wrapper.py\live_portrait_wrapper.py]8;;\:]8;id=510750;file:///content/LivePortrait/src/live_portrait_wrapper.py#46\46]8;;\
           /content/LivePortrait/pretrained_weights/                            
           liveportrait/base_models/appearance_featu                            
           re_extractor.pth done.                                               
           Load motion_extractor from                ]8;id=61461;file:///content/LivePortrait/src/live_portrait_wrapper.py\live_portrait_wrapper.py]8;;\:]8;id=409166;file:///content/LivePortrait/src/live_portrait_wrapper.py#49\49]8;;\
           /content/LivePortrait/pretrained_weights/                            
           liveportrait/base_models/motion_extractor                            
           .pth done.                                                           
[20:4

In [10]:
%cd /content/LivePortrait
!ls -l assets/examples/driving/

/content/LivePortrait
total 15824
-rw-r--r-- 1 root root   25712 Aug  4 19:57 aggrieved.pkl
-rw-r--r-- 1 root root 2958395 Aug  4 19:57 d0.mp4
-rw-r--r-- 1 root root   86025 Aug  4 20:37 d0.pkl
-rw-r--r-- 1 root root 1064930 Aug  4 19:57 d10.mp4
-rw-r--r-- 1 root root  468504 Aug  4 19:57 d11.mp4
-rw-r--r-- 1 root root   98910 Aug  4 19:57 d12.jpg
-rw-r--r-- 1 root root  596446 Aug  4 19:57 d12.mp4
-rw-r--r-- 1 root root 2475854 Aug  4 19:57 d13.mp4
-rw-r--r-- 1 root root  891025 Aug  4 19:57 d14.mp4
-rw-r--r-- 1 root root  187263 Aug  4 19:57 d18.mp4
-rw-r--r-- 1 root root   68716 Aug  4 19:57 d19.jpg
-rw-r--r-- 1 root root  232859 Aug  4 19:57 d19.mp4
-rw-r--r-- 1 root root    8599 Aug  4 19:57 d1.pkl
-rw-r--r-- 1 root root  462335 Aug  4 19:57 d20.mp4
-rw-r--r-- 1 root root    8599 Aug  4 19:57 d2.pkl
-rw-r--r-- 1 root root   77182 Aug  4 19:57 d30.jpg
-rw-r--r-- 1 root root   75277 Aug  4 19:57 d38.jpg
-rw-r--r-- 1 root root 1430968 Aug  4 19:57 d3.mp4
-rw-r--r-- 1 root root   7777

In [14]:
from glob import glob
from IPython.display import Video, display
import os # Added explicit import for os

# Specifically look for the animation generated with d9.mp4
output_path = '/content/LivePortrait/animations/source--d9_concat.mp4'

print(f'Checking for animation file: {output_path}')

if not os.path.exists(output_path):
    # If the specific file is not found, try the general glob approach as a fallback
    outputs = sorted(glob('/content/LivePortrait/animations/*.mp4'))

    print(f"Glob found: {outputs}") # Explicitly print the result of glob

    if not outputs:
        # Added a diagnostic print to see what glob finds
        print('No MP4 files found in /content/LivePortrait/animations/. Current directory:', os.getcwd())
        # Ensure the directory exists before listing
        if os.path.exists('/content/LivePortrait/animations'):
            print('Contents of animations directory:', os.listdir('/content/LivePortrait/animations'))
        else:
            print('Directory /content/LivePortrait/animations does not exist.')
        raise FileNotFoundError('LivePortrait did not produce an MP4 file.')
    output_path = outputs[-1]
    print('Result (from glob):', output_path)
else:
    print('Result:', output_path)

display(Video(output_path, embed=True, width=512))

Checking for animation file: /content/LivePortrait/animations/source--d9_concat.mp4
Result: /content/LivePortrait/animations/source--d9_concat.mp4


In [12]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>